# Missing Data Imputation

Missing data is a common challenge in statistical analysis. Traditional approaches like listwise deletion (removing rows with missing values) can lead to biased estimates and reduced statistical power. Bayesian methods offer a principled approach to handle missing data through imputation, where missing values are treated as parameters to be estimated from the posterior distribution.

Bambi supports missing data imputation using the `mi()` function, similar to the approach used in [brms](https://paul-buerkner.github.io/brms/). This functionality leverages PyMC's automatic imputation mechanism to handle missing values in both response and predictor variables.

## Setup

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import arviz as az
import bambi as bmb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
az.style.use("arviz-darkgrid")
SEED = 42
rng = np.random.default_rng(SEED)

## The `mi()` Function

The `mi()` function (short for "missing imputation") is used to mark variables that have missing values (represented as `NaN` in pandas/numpy) that should be imputed during model fitting. When Bambi encounters a variable wrapped with `mi()`, it:

1. Preserves the `NaN` values instead of dropping rows
2. Creates latent variables for the missing values
3. Samples these latent variables from their posterior distribution during MCMC

The `mi()` function can be used on:
- **Response variables**: `mi(y) ~ x` - imputes missing values in the outcome
- **Predictor variables**: `y ~ mi(x)` - imputes missing values in covariates

## Simulating Data with Missing Values

Let's create a simple dataset where we know the true data generating process, then introduce missing values to demonstrate imputation.

In [ ]:
# Generate complete data
n = 100
true_intercept = 2.0
true_slope = 3.0
true_sigma = 1.0

x_complete = rng.normal(0, 1, size=n)
y_complete = true_intercept + true_slope * x_complete + rng.normal(0, true_sigma, size=n)

# Create DataFrame with complete data
data_complete = pd.DataFrame({"x": x_complete, "y": y_complete})

print(f"Complete data shape: {data_complete.shape}")
print(f"\nFirst few rows:")
data_complete.head()

Now let's introduce missing values. We'll create two scenarios:
1. Missing values in the response (y)
2. Missing values in the predictor (x)

In [ ]:
# Introduce missing values in response (Missing Completely At Random - MCAR)
n_missing_y = 15
missing_idx_y = rng.choice(n, size=n_missing_y, replace=False)

y_with_missing = y_complete.copy()
y_with_missing[missing_idx_y] = np.nan

data_missing_response = pd.DataFrame({"x": x_complete, "y": y_with_missing})

print(f"Data with missing response:")
print(f"  Total rows: {len(data_missing_response)}")
print(f"  Missing y values: {data_missing_response['y'].isna().sum()}")
print(f"  Complete cases: {data_missing_response.dropna().shape[0]}")

In [ ]:
# Introduce missing values in predictor (MCAR)
n_missing_x = 15
missing_idx_x = rng.choice(n, size=n_missing_x, replace=False)

x_with_missing = x_complete.copy()
x_with_missing[missing_idx_x] = np.nan

data_missing_predictor = pd.DataFrame({"x": x_with_missing, "y": y_complete})

print(f"Data with missing predictor:")
print(f"  Total rows: {len(data_missing_predictor)}")
print(f"  Missing x values: {data_missing_predictor['x'].isna().sum()}")
print(f"  Complete cases: {data_missing_predictor.dropna().shape[0]}")

## Response Imputation: `mi(y) ~ x`

When the response variable has missing values, we can use `mi()` to impute them. The missing response values are sampled from the posterior predictive distribution, which is determined by the likelihood and the linear predictor.

### Traditional Approach: Listwise Deletion

First, let's see what happens with the traditional approach of dropping rows with missing values:

In [ ]:
# Model with listwise deletion (dropna=True)
model_listwise = bmb.Model("y ~ x", data_missing_response, dropna=True)
results_listwise = model_listwise.fit(draws=1000, tune=1000, chains=2, random_seed=SEED)

In [ ]:
az.summary(results_listwise, var_names=["Intercept", "x", "sigma"])

### Bayesian Imputation with `mi()`

Now let's use `mi()` to impute the missing response values:

In [ ]:
# Model with missing data imputation
model_impute_y = bmb.Model("mi(y) ~ x", data_missing_response)
model_impute_y

Notice that Bambi reports the full number of observations (100), not just the complete cases. The `is_mi` flag on the response term indicates that missing values will be imputed.

In [ ]:
# Fit the model
results_impute_y = model_impute_y.fit(draws=1000, tune=1000, chains=2, random_seed=SEED)

In [ ]:
az.summary(results_impute_y, var_names=["Intercept", "x", "sigma"])

### Comparing Results

Let's compare the parameter estimates from both approaches:

In [ ]:
# Compare parameter estimates
comparison_data = {
    "True Value": [true_intercept, true_slope, true_sigma],
    "Listwise Deletion": [
        float(results_listwise.posterior["Intercept"].mean()),
        float(results_listwise.posterior["x"].mean()),
        float(results_listwise.posterior["sigma"].mean()),
    ],
    "MI Imputation": [
        float(results_impute_y.posterior["Intercept"].mean()),
        float(results_impute_y.posterior["x"].mean()),
        float(results_impute_y.posterior["sigma"].mean()),
    ],
}

comparison_df = pd.DataFrame(comparison_data, index=["Intercept", "Slope (x)", "Sigma"])
comparison_df

### Examining Imputed Values

One advantage of Bayesian imputation is that we get a full posterior distribution for each imputed value, not just a point estimate. This allows us to quantify the uncertainty in our imputations.

In [ ]:
# The imputed values are stored in the posterior
# They correspond to the missing observations
print(f"Number of imputed values: {n_missing_y}")
print(f"\nTrue values at missing positions:")
print(y_complete[missing_idx_y][:5])  # First 5

# Get the imputed values from the posterior
if "mi(y)" in results_impute_y.posterior:
    imputed_y = results_impute_y.posterior["mi(y)"]
    print(f"\nPosterior mean of imputed values (first 5):")
    print(imputed_y.mean(dim=["chain", "draw"]).values[:5])

## Predictor Imputation: `y ~ mi(x)`

When a predictor variable has missing values, we can also use `mi()` to impute them. This creates a sub-model for the predictor where:
- Observed values of the predictor constrain the distribution
- Missing values are sampled from the posterior of the predictor distribution
- The imputed predictor values are then used in the main regression model

In [ ]:
# Model with missing predictor imputation
model_impute_x = bmb.Model("y ~ mi(x)", data_missing_predictor)
model_impute_x

In [ ]:
# Fit the model
results_impute_x = model_impute_x.fit(draws=1000, tune=1000, chains=2, random_seed=SEED)

In [ ]:
az.summary(results_impute_x, var_names=["Intercept", "mi(x)", "sigma"])

### Examining Imputed Predictor Values

The imputed predictor values are available in the posterior:

In [ ]:
# Compare imputed x values to true values
if "mi(x)_imputed" in results_impute_x.posterior:
    imputed_x = results_impute_x.posterior["mi(x)_imputed"]
    imputed_x_mean = imputed_x.mean(dim=["chain", "draw"]).values
    
    # Get the true values at missing positions
    true_x_at_missing = x_complete[missing_idx_x]
    
    print("Comparison of true vs imputed x values:")
    print(f"{'True':>10} {'Imputed Mean':>15}")
    print("-" * 28)
    for true_val, imp_val in zip(true_x_at_missing[:5], imputed_x_mean[missing_idx_x][:5]):
        print(f"{true_val:>10.3f} {imp_val:>15.3f}")

## Comparison with Complete Data

Let's fit a model on the complete data (before introducing missing values) to see how well our imputation approaches recover the true relationships:

In [ ]:
# Model on complete data
model_complete = bmb.Model("y ~ x", data_complete)
results_complete = model_complete.fit(draws=1000, tune=1000, chains=2, random_seed=SEED)

In [ ]:
# Compare all approaches
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

params = ["Intercept", "x", "sigma"]
true_values = [true_intercept, true_slope, true_sigma]

for ax, param, true_val in zip(axes, params, true_values):
    # Handle the case where the parameter name might be different
    param_name_impute_x = "mi(x)" if param == "x" else param
    
    # Get posterior samples
    samples_complete = results_complete.posterior[param].values.flatten()
    samples_listwise = results_listwise.posterior[param].values.flatten()
    samples_impute_y = results_impute_y.posterior[param].values.flatten()
    
    if param_name_impute_x in results_impute_x.posterior:
        samples_impute_x = results_impute_x.posterior[param_name_impute_x].values.flatten()
    else:
        samples_impute_x = results_impute_x.posterior[param].values.flatten()
    
    # Plot
    ax.axvline(true_val, color="red", linestyle="--", label="True value", linewidth=2)
    ax.hist(samples_complete, bins=30, alpha=0.5, density=True, label="Complete data")
    ax.hist(samples_listwise, bins=30, alpha=0.5, density=True, label="Listwise deletion")
    ax.hist(samples_impute_y, bins=30, alpha=0.5, density=True, label="MI (response)")
    ax.set_xlabel(param)
    ax.set_title(f"Posterior: {param}")

axes[0].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## When to Use Missing Data Imputation

Bayesian imputation with `mi()` is particularly useful when:

1. **You have informative missingness**: The observed data can help predict missing values
2. **You want to avoid information loss**: Listwise deletion discards entire rows
3. **You need uncertainty quantification**: Point imputation methods don't capture uncertainty
4. **Missing data is MCAR or MAR**: The method assumes data is Missing Completely At Random or Missing At Random

### Assumptions and Limitations

- **Missing At Random (MAR)**: The imputation assumes that missingness depends only on observed data
- **Distributional assumptions**: For predictor imputation, a Normal distribution is assumed by default
- **Computational cost**: Imputation adds parameters to the model, increasing computation time
- **Model specification**: The imputation model should be correctly specified

## Summary

Bambi's `mi()` function provides a convenient way to handle missing data through Bayesian imputation:

| Usage | Description |
|:------|:------------|
| `mi(y) ~ x` | Impute missing values in the response variable |
| `y ~ mi(x)` | Impute missing values in a predictor variable |
| `mi(y) ~ mi(x)` | Impute missing values in both response and predictor |

Key benefits:
- Uses all available data instead of discarding incomplete cases
- Properly propagates uncertainty from imputation to parameter estimates
- Provides posterior distributions for imputed values
- Integrates seamlessly with Bambi's formula interface

In [ ]:
%load_ext watermark
%watermark -n -u -v -iv -w